# Hands On with Advanced AI & Emerging Applications

# Retrieval Augmented Generation

RAG (Retrieval-Augmented Generation) is a technique that is used in natural language processing to enhance the capabilities of Large Language Models (LLMs) by integrating information retrieval. It improves the responses of LLMs by incorporating updated or user-defined data the LLM was originally not trained on. A RAG system is typically composed of two processes: Indexing and Retrieval + Generation.

This module uses [LangChain](https://www.langchain.com/) to interact with LLMs. LangChain is a framework designed to facilitate the development of applications powered by LLMs. It provides an abstraction layer by decomposing applications as a set of _chains_ made of prompts, models, and data sources. LangChain offers libraries to interact with most of the foundational models available in the market.

In this module, we will use [Amazon Bedrock](https://aws.amazon.com/bedrock/), which is a fully managed service from AWS that offers access to several foundational models. We have chosen [Claude Sonnet](https://www.anthropic.com/claude/sonnet) from Anthropic as the LLM to use.

### Setting up connection to Amazon Bedrock

In [ ]:
pip install -q langchain-community langchain_aws

In [ ]:
from langchain_aws import ChatBedrock

llm = ChatBedrock(
    model_id="us.meta.llama3-3-70b-instruct-v1:0",
    model_kwargs=dict(temperature=0),
    region="us-east-1"
)

In [ ]:
ai_msg = llm.invoke("Who is the president of the USA?")
print(ai_msg.content)

**Note**: See how Llama 3.3 70B, an LLM released in December 2024, was trained on data that is no longer accurate.

### Indexing

The indexing phase is responsible for organizing data so that it can be efficiently retrieved by applications powered by large language models (LLMs). To scale Retrieval-Augmented Generation (RAG) systems and handle large datasets, vector databases are commonly used. These databases store information and facilitate efficient data indexing and querying

For the purpose of this section, we are going to load the content of the markdown file `acme_guidelines.md`. This document has guidelines from the fictional company `ACME` about the IP address schemes to be used in device configuration.


LangChain offers a vast number of classes to load documents, ranging from text files (`.txt`) to HTML and PDF documents. To load the `acme_guidelines.md` file, we are going to use the class `TextLoader` from the `langchain.document_loaders` library.


In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("acme_guidelines.md")
documents = loader.load()

#### Splitting
As a best practice, data to be stored in the Vector DB is usually split into smaller _chunks_ of data. This makes the _querying_ stage more efficient, as only relevant pieces of the data are retrieved. Additionally, large chunks may later not fit in the LLM context window. 

There are many strategies to split a text. For example, to consider special characters (HTML tags) or punctuation signs. In our example, we are using yet another LangChain class to split a text. The `MarkdownHeaderTextSplitter` allows us to split markdown files based on _Headers_ or _sections_.



In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

md_splits = md_splitter.split_text(documents[0].page_content)
len(md_splits)

In [ ]:
print(md_splits[1].page_content)

Another alternative to split text files is to use the class `RecursiveCharacterTextSplitter`, which allows us to define separators between chunks, the size of each chunk and the number of items being overlapped between chunks.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"], 
    chunk_size=200,
    chunk_overlap=0,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(documents)
len(all_splits)

At this point we can test the content of a chunk by displaying the attribute `page_content`

In [ ]:
print(all_splits[1].page_content)

#### Embedding

The next and last stage in the _Indexing_ process is to actually store the chunks in a Vector DB.

As the name implies, a Vector DB stores data as **vectors**, not as raw text; therefore, an embedding function must transform these chunks into vectors. In this task, we are using an _In-Memory_ Database and [Amazon Titan Embeddings](https://docs.aws.amazon.com/bedrock/latest/userguide/titan-embedding-models.html).

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_aws import BedrockEmbeddings

In [ ]:
embeddings = BedrockEmbeddings(
    model_id="amazon.titan-embed-text-v2:0",
    region_name='us-east-1'
)

In [ ]:
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(documents=md_splits)

Note that we have used the variable `md_splits`, which has the chunks generated by the `MarkdownHeaderTextSplitter`.

Let us examine the first two entries in the Vector DB. Note how the dimension of the embeddings is 1024. Only the first three items of each vector are being displayed.

In [ ]:
for doc in vector_store.store.values():
    print(f"{doc['vector'][:3]}... (dim={len(doc['vector'])})")
    print(f"{doc['text'][:40]}...")
    print()


#### Similarity Search

Before actually implementing the retrieval stage with the LLM, let us test the _querying_ of the vector DB with the method `similarity_search`.

In [ ]:
query = "approved security mechanisms"
results = vector_store.similarity_search(query, k=2)
for result in results:
    print("Metadata:", result.metadata)
    print("Content:")
    print(result.page_content)
    print("-"*160)

**Note:** See how the similarity search takes into account the semantic meaning of the text rather than exact words or grammar.

### Retrieval Using LangChain LCEL

In this task, we are going to use LangChain constructs, namely _chains_, to test the capabilities of the RAG system. As a first step, we will invoke the LLM without using RAG just to realize the difference.

In [ ]:
from langchain_core.runnables.passthrough import RunnablePassthrough
from langchain_core.prompts import PromptTemplate

query = "What IPs are allowed in Ethernet Interfaces?"

#### Query without RAG

In [ ]:
result = llm.invoke(query)
print(result.content)

See how, using a basic prompt and invoking the LLM directly, the generated response is very generic. The LLM claims that any IP address can be used on an Ethernet Interface.

#### Querying with RAG

The idea behind RAG is to provide more context to the LLM so it can use it to generate a response. In our current example, the context is the policies listed in the `acme_guidelines.md` document. We will use a prompt that, apart from specifying the question to the LLM, also gives context to it.
```
HUMAN

You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

Question: {question} 

Context: {context} 

Answer:
```

In [ ]:
retrieval_qa_chat_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
        You are a network engineer assistant. Use the following documentation to answer the question.'
        If the answer is not in the documentation, say you don’t know.

        Documentation:
        {context}

        Question: {question}

        Answer:
"""
)

Note how the `rag_chain` is made from three components:

1. Two parallel branches to get (a) the `context` and (b) the `question`
2. The RAG prompt
3. The LLM

In [ ]:
def retrieve_docs(question):
    docs = vector_store.similarity_search(question, k=2)
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
  
rag_chain = ({'context': retrieve_docs, 'question' : RunnablePassthrough()} | retrieval_qa_chat_prompt | llm )

result = rag_chain.invoke("What IPs are allowed on Ethernet Interfaces?")
print(result.content)


See how in this execution, the LLM considered the content of the `acme_guidelines.md` markdown file.